# TransitPulse EDA

This notebook connects to the TransitPulse PostgreSQL database, queries the star schema, and saves EDA charts as PNG files in `notebooks/charts/`.

In [ ]:
from pathlib import Path
from os import getenv
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12

cwd = Path.cwd()
env_path = cwd / ".env"
if not env_path.exists():
    env_path = cwd.parent / ".env"

if not env_path.exists():
    raise FileNotFoundError("Could not find .env in the current directory or parent directory.")

load_dotenv(env_path)
PROJECT_ROOT = env_path.parent
CHART_DIR = PROJECT_ROOT / "notebooks" / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

required_vars = ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
missing = [name for name in required_vars if not getenv(name)]
if missing:
    raise RuntimeError(f"Missing required .env variables: {', '.join(missing)}")

user = quote_plus(getenv("DB_USER", ""))
password = quote_plus(getenv("DB_PASSWORD", ""))
host = getenv("DB_HOST")
port = getenv("DB_PORT")
database = getenv("DB_NAME")
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_string)

def read_query(sql: str) -> pd.DataFrame:
    return pd.read_sql(text(sql), engine)

def save_chart(filename: str) -> None:
    output_path = CHART_DIR / filename
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {output_path}")

print(f"Charts will be saved to: {CHART_DIR}")

## Daily Ridership Trend

In [ ]:
daily_ridership = read_query(
    """
    SELECT
        d.full_date,
        SUM(f.passenger_count) AS total_passengers
    FROM fact_ridership f
    JOIN dim_date d ON f.date_id = d.date_id
    GROUP BY d.full_date
    ORDER BY d.full_date
    """
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=daily_ridership, x="full_date", y="total_passengers", linewidth=2)
plt.title("Daily Ridership Trend")
plt.xlabel("Date")
plt.ylabel("Total Passengers")
plt.xticks(rotation=45)
save_chart("daily_ridership_trend.png")

## Ridership by Route

In [ ]:
route_ridership = read_query(
    """
    SELECT
        COALESCE(NULLIF(r.route_name, ''), r.route_code) AS route,
        SUM(f.passenger_count) AS total_passengers
    FROM fact_ridership f
    JOIN dim_route r ON f.route_id = r.route_id
    GROUP BY route
    ORDER BY total_passengers DESC
    """
)

plt.figure(figsize=(14, 7))
sns.barplot(data=route_ridership, x="route", y="total_passengers", color="#3973ac")
plt.title("Ridership by Route")
plt.xlabel("Route")
plt.ylabel("Total Passengers")
plt.xticks(rotation=45, ha="right")
save_chart("ridership_by_route.png")

## Hourly Ridership Heatmap

In [ ]:
hourly_heatmap = read_query(
    """
    SELECT
        d.weekday,
        EXTRACT(ISODOW FROM d.full_date)::INT AS weekday_order,
        t.hour,
        SUM(f.passenger_count) AS total_passengers
    FROM fact_ridership f
    JOIN dim_date d ON f.date_id = d.date_id
    JOIN dim_time t ON f.time_id = t.time_id
    GROUP BY d.weekday, weekday_order, t.hour
    ORDER BY weekday_order, t.hour
    """
)

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heatmap_data = hourly_heatmap.pivot(index="weekday", columns="hour", values="total_passengers")
heatmap_data = heatmap_data.reindex(weekday_order)

plt.figure(figsize=(15, 6))
sns.heatmap(heatmap_data, cmap="YlGnBu", linewidths=0.3, linecolor="white")
plt.title("Hourly Ridership Heatmap")
plt.xlabel("Hour of Day")
plt.ylabel("Weekday")
save_chart("hourly_ridership_heatmap.png")

## Top 20 Busiest Stops

In [ ]:
top_stops = read_query(
    """
    SELECT
        s.stop_name,
        SUM(f.passenger_count) AS total_passengers
    FROM fact_ridership f
    JOIN dim_stop s ON f.stop_id = s.stop_id
    GROUP BY s.stop_name
    ORDER BY total_passengers DESC
    LIMIT 20
    """
)

plt.figure(figsize=(12, 9))
sns.barplot(data=top_stops, x="total_passengers", y="stop_name", color="#2f855a")
plt.title("Top 20 Busiest Stops")
plt.xlabel("Total Passengers")
plt.ylabel("Stop")
save_chart("top_20_busiest_stops.png")

## Average Delay by Route

In [ ]:
avg_delay_by_route = read_query(
    """
    SELECT
        COALESCE(NULLIF(r.route_name, ''), r.route_code) AS route,
        AVG(f.delay_minutes) AS avg_delay_minutes
    FROM fact_ridership f
    JOIN dim_route r ON f.route_id = r.route_id
    GROUP BY route
    ORDER BY avg_delay_minutes DESC
    """
)

plt.figure(figsize=(14, 7))
sns.barplot(data=avg_delay_by_route, x="route", y="avg_delay_minutes", color="#b45309")
plt.title("Average Delay by Route")
plt.xlabel("Route")
plt.ylabel("Average Delay (minutes)")
plt.xticks(rotation=45, ha="right")
save_chart("average_delay_by_route.png")

## Peak vs Off-Peak Ridership

In [ ]:
peak_comparison = read_query(
    """
    SELECT
        CASE WHEN t.is_peak_hour THEN 'Peak' ELSE 'Off-Peak' END AS period_type,
        SUM(f.passenger_count) AS total_passengers
    FROM fact_ridership f
    JOIN dim_time t ON f.time_id = t.time_id
    GROUP BY period_type
    ORDER BY period_type
    """
)

plt.figure(figsize=(8, 8))
plt.pie(
    peak_comparison["total_passengers"],
    labels=peak_comparison["period_type"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#1f77b4", "#ff7f0e"],
)
plt.title("Peak vs Off-Peak Ridership")
plt.axis("equal")
save_chart("peak_vs_off_peak_ridership.png")

In [ ]:
engine.dispose()